# PNS CoT Optimization — Results Notebook

Loads **pre-computed** CoT and PNS-optimized chains from the paper's data files and displays the comparison table.

**No model generation required** — all results come from the saved JSONL files.

| File | Content |
|---|---|
| `gsm8k_cot_qwen3-14b-thinking-v2.jsonl` | Original CoT responses (Qwen3-14B) |
| `gsm8k_pns_qwen3-14b-thinking-v2.jsonl` | PNS-optimized chains + per-step PN values |
| `test_math500_answered_qwen3-14B-thinking-v2.jsonl` | Original CoT responses (Qwen3-14B) |
| `math500_pns_qwen3-v2-pb-fixed.jsonl` | PNS-optimized chains + per-step PN values |

**Algorithm 1 — PNS Step Pruning (recap):**
- For each step $s_i$: generate replacement → roll out $k$ completions → $\text{PN}(s_i) = 1 - \text{avg\_correct}$
- Prune if $\text{PN}(s_i) < \alpha = 0.5$ (step not causally necessary)
- $\text{PS(chain)} = 1$ iff final answer is correct

## 1 · Upload data files

Upload all four JSONL files when prompted.

In [ ]:
from google.colab import files
import json, re, os

print("Upload the 4 JSONL files:")
print("  1. gsm8k_cot_qwen3-14b-thinking-v2.jsonl")
print("  2. gsm8k_pns_qwen3-14b-thinking-v2.jsonl")
print("  3. test_math500_answered_qwen3-14B-thinking-v2.jsonl")
print("  4. math500_pns_qwen3-v2-pb-fixed.jsonl")
uploaded = files.upload()
print(f"\nUploaded: {list(uploaded.keys())}")

## 2 · Load files

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


# ── File paths (gdown places them in /content/)
GSM_COT_FILE  = "gsm8k_cot_qwen3-14b-thinking-v2.jsonl"
GSM_PNS_FILE  = "gsm8k_pns_qwen3-14b-thinking-v2.jsonl"
MATH_COT_FILE = "test_math500_answered_qwen3-14B-thinking-v2.jsonl"
MATH_PNS_FILE = "math500_pns_qwen3-v2-pb-fixed.jsonl"

gsm_cot  = load_jsonl(GSM_COT_FILE)
gsm_pns  = load_jsonl(GSM_PNS_FILE)
math_cot = load_jsonl(MATH_COT_FILE)
math_pns = load_jsonl(MATH_PNS_FILE)

print(f"GSM8K   COT : {len(gsm_cot):4d} records")
print(f"GSM8K   PNS : {len(gsm_pns):4d} records")
print(f"MATH-500 COT: {len(math_cot):4d} records")
print(f"MATH-500 PNS: {len(math_pns):4d} records")

## 3 · Match COT ↔ PNS records and pick 10 samples

Match by question text. Pick 8 GSM8K + 2 MATH-500 samples where PS(chain)=1 (correct original chain).

In [ ]:
def build_lookup(records):
    """Index records by question text for fast matching."""
    return {r.get("question", r.get("problem", "")).strip(): r for r in records}


def merge_records(cot_records, pns_records, dataset, n):
    """
    Join COT and PNS records on question. Pick first n where PS(chain)=1.
    Returns list of merged dicts.
    """
    pns_lookup = build_lookup(pns_records)
    merged = []
    for cot in cot_records:
        q = cot.get("question", cot.get("problem", "")).strip()
        pns = pns_lookup.get(q)
        if pns is None:
            continue
        m = pns.get("metrics", {})
        if m.get("PS(chain)", 0) != 1:
            continue   # skip PS=0 (incorrect original chain)

        orig_cot   = cot.get("model_answer", "")
        pns_chain  = m.get("final_chain", [])
        pns_text   = "\n\n".join(pns_chain)

        merged.append({
            "dataset":        dataset,
            "question":       q,
            "answer":         str(pns.get("answer", cot.get("answer", ""))),
            # Original CoT
            "orig_cot":       orig_cot,
            "orig_tokens":    m.get("original_token_length", len(orig_cot.split())),
            "orig_steps":     m.get("original_step_length",  len([s for s in orig_cot.split("\n\n") if s.strip()])),
            "orig_ps":        m.get("original_accuracy",     1),
            # PNS-optimized
            "pns_chain":      pns_chain,
            "pns_text":       pns_text,
            "pns_tokens":     m.get("token_length",   len(pns_text.split())),
            "pns_steps":      m.get("step_length",    len(pns_chain)),
            "pns_ps":         m.get("PS(chain)",      1),
            # PN metrics
            "pn_per_step":    m.get("pn_per_step",    []),
            "avg_pn":         m.get("avg_PN(steps)",  None),
            "max_pn":         m.get("max_PN(steps)",  None),
            "min_pn":         m.get("min_PN(steps)",  None),
        })
        if len(merged) == n:
            break
    return merged


gsm_samples  = merge_records(gsm_cot,  gsm_pns,  "gsm8k",   n=8)
math_samples = merge_records(math_cot, math_pns, "math500", n=2)
samples      = gsm_samples + math_samples

print(f"Matched: {len(gsm_samples)} GSM8K  +  {len(math_samples)} MATH-500  =  {len(samples)} total")
print()
for i, s in enumerate(samples, 1):
    tok_red = (s['orig_tokens'] - s['pns_tokens']) / max(s['orig_tokens'], 1) * 100
    print(f"  [{i:02d}] {s['dataset']:7s}  "
          f"orig={s['orig_steps']}steps/{s['orig_tokens']}tok  "
          f"pns={s['pns_steps']}steps/{s['pns_tokens']}tok  "
          f"(-{tok_red:.0f}%)  "
          f"ans={s['answer'][:12]!r}  "
          f"{s['question'][:50]}")

## 4 · Answer extraction helpers

In [ ]:
def _extract_boxed(text):
    results, start = [], 0
    while True:
        idx = text.find(r"\boxed{", start)
        if idx == -1: break
        depth = 0
        for i in range(idx + 7, len(text)):
            if text[i] == "{": depth += 1
            elif text[i] == "}":
                if depth == 0: results.append(text[idx+7:i]); start = i+1; break
                depth -= 1
        else: break
    return results[-1].strip() if results else ""

def _normalize(t):
    t = re.sub(r"\\left|\\right|\\,|\\!", "", t)
    t = re.sub(r"\^\\circ|\\circ|°", "", t)
    t = re.sub(r"\\text\{([^}]*)\}", r"\1", t)
    t = re.sub(r"\\d?frac", r"\\frac", t)
    t = re.sub(r"\$+|\s+", "", t)
    return t.lower().strip()

def answers_equivalent(a, b):
    if not a or not b: return False
    if _normalize(a) == _normalize(b): return True
    try: return float(a.replace(",","")) == float(str(b).replace(",",""))
    except: return False

def extract_answer(text, dataset):
    boxed = _extract_boxed(text)
    if boxed: return boxed
    if "gsm" in dataset.lower():
        m = re.search(r"####\s*([\d,.\-]+)", text)
        if m: return m.group(1).replace(",","").strip()
    return ""

print("Helpers ready.")

## 5 · Comparison table (paper format)

Original CoT (Qwen3-14B) vs PNS-optimized chain — tokens, steps, reduction, PS, and avg PN.

In [ ]:
PNS_THRESHOLD = 0.5

print("Model: Qwen/Qwen3-14B  |  PNS alpha=0.5  |  k=3 rollouts per step")
sep = "-" * 108
print(sep)
print(f"{'#':>2}  {'Dataset':<8} {'Orig Tok':>8} {'PNS Tok':>7} {'Tok Red%':>8} "
      f"{'Orig Steps':>10} {'PNS Steps':>9} {'Step Red%':>9} "
      f"{'PS(orig)':>8} {'PS(pns)':>7} {'Avg PN':>7} {'Correct':>7}")
print(sep)

t_otok=t_ptok=t_ostep=t_pstep=t_ops=t_pps=t_correct=0
pn_all = []

for i, s in enumerate(samples, 1):
    tok_red  = (s['orig_tokens'] - s['pns_tokens']) / max(s['orig_tokens'],  1) * 100
    step_red = (s['orig_steps']  - s['pns_steps'])  / max(s['orig_steps'],   1) * 100

    # Verify correctness from PNS chain
    pns_last     = s['pns_chain'][-1] if s['pns_chain'] else ""
    pns_extracted = extract_answer(pns_last, s['dataset'])
    correct      = int(answers_equivalent(pns_extracted, s['answer']))

    avg_pn_str = f"{s['avg_pn']:.3f}" if s['avg_pn'] is not None else "  N/A"

    print(f"{i:>2}  {s['dataset']:<8} {s['orig_tokens']:>8} {s['pns_tokens']:>7} "
          f"{tok_red:>7.1f}%  {s['orig_steps']:>10} {s['pns_steps']:>9} "
          f"{step_red:>8.1f}%  {s['orig_ps']:>8}  {s['pns_ps']:>6}  "
          f"{avg_pn_str:>7}  {'Y' if correct else 'N':>7}")

    t_otok  += s['orig_tokens'];  t_ptok  += s['pns_tokens']
    t_ostep += s['orig_steps'];   t_pstep += s['pns_steps']
    t_ops   += s['orig_ps'];      t_pps   += s['pns_ps']
    t_correct += correct
    if s['avg_pn'] is not None: pn_all.append(s['avg_pn'])

n = len(samples)
avg_tok_red  = (t_otok  - t_ptok)  / max(t_otok,  1) * 100
avg_step_red = (t_ostep - t_pstep) / max(t_ostep, 1) * 100
avg_pn_all   = sum(pn_all)/len(pn_all) if pn_all else None
pn_str       = f"{avg_pn_all:.3f}" if avg_pn_all else "  N/A"

print(sep)
print(f"{'AVG':>2}  {'':8} {t_otok/n:>8.0f} {t_ptok/n:>7.0f} "
      f"{avg_tok_red:>7.1f}%  {t_ostep/n:>10.1f} {t_pstep/n:>9.1f} "
      f"{avg_step_red:>8.1f}%  {t_ops/n:>8.2f}  {t_pps/n:>6.2f}  "
      f"{pn_str:>7}  {t_correct}/{n:>5}")
print(sep)
print(f"\nSummary:")
print(f"  Original CoT  : acc={t_ops/n*100:.1f}%  avg_steps={t_ostep/n:.1f}  avg_tokens={t_otok/n:.0f}")
print(f"  PNS-optimized : acc={t_pps/n*100:.1f}%  avg_steps={t_pstep/n:.1f}  avg_tokens={t_ptok/n:.0f}")
print(f"  Token reduction : {avg_tok_red:.1f}%   |   Step reduction: {avg_step_red:.1f}%")
if avg_pn_all: print(f"  Mean PN across all steps: {avg_pn_all:.3f}")

## 6 · Per-step PN breakdown

In [ ]:
print(f"Per-step PN values  (alpha={PNS_THRESHOLD} — below = pruned)\n")
for i, s in enumerate(samples, 1):
    print(f"[{i:02d}] {s['dataset']:7s}  {s['question'][:70]}")
    if not s['pn_per_step']:
        print("      No PN values recorded")
    else:
        for si, pn in enumerate(s['pn_per_step'], 1):
            tag = f"PRUNED  PN={pn:.3f}" if pn < PNS_THRESHOLD else f"kept    PN={pn:.3f}"
            print(f"      step {si}: {tag}")
    print(f"      Pruned chain ({s['pns_steps']} steps):")
    for si, node in enumerate(s['pns_chain'], 1):
        print(f"        [{si}] {node[:90]}{'...' if len(node)>90 else ''}")
    print()

## 7 · Side-by-side: Original CoT vs PNS-pruned chain

In [ ]:
for i, s in enumerate(samples, 1):
    sep = "=" * 80
    tok_red  = (s['orig_tokens'] - s['pns_tokens']) / max(s['orig_tokens'], 1) * 100
    step_red = (s['orig_steps']  - s['pns_steps'])  / max(s['orig_steps'],  1) * 100
    print(sep)
    print(f"[{i:02d}] {s['dataset'].upper()}  |  Expected: {s['answer']}")
    print(f"Q: {s['question'][:100]}{'...' if len(s['question'])>100 else ''}")
    print(sep)

    print(f"\n--- ORIGINAL CoT ({s['orig_steps']} steps, {s['orig_tokens']} tokens) ---")
    orig_nodes = [nd.strip() for nd in s['orig_cot'].split("\n\n") if nd.strip()]
    for si, node in enumerate(orig_nodes, 1):
        print(f"  [{si}] {node[:100]}{'...' if len(node)>100 else ''}")

    print(f"\n--- PNS-PRUNED ({s['pns_steps']} steps, {s['pns_tokens']} tokens  "
          f"-{tok_red:.0f}% tokens, -{step_red:.0f}% steps) ---")
    for si, node in enumerate(s['pns_chain'], 1):
        print(f"  [{si}] {node[:100]}{'...' if len(node)>100 else ''}")
    print()

## 8 · Save results

In [ ]:
OUT_PATH = "/content/pns_comparison_results.jsonl"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for i, s in enumerate(samples, 1):
        tok_red  = (s['orig_tokens'] - s['pns_tokens']) / max(s['orig_tokens'], 1) * 100
        step_red = (s['orig_steps']  - s['pns_steps'])  / max(s['orig_steps'],  1) * 100
        pns_last = s['pns_chain'][-1] if s['pns_chain'] else ""
        pns_extracted = extract_answer(pns_last, s['dataset'])
        f.write(json.dumps({
            "idx": i, "dataset": s["dataset"], "question": s["question"], "answer": s["answer"],
            "orig_tokens": s["orig_tokens"], "orig_steps": s["orig_steps"], "orig_ps": s["orig_ps"],
            "pns_tokens":  s["pns_tokens"],  "pns_steps":  s["pns_steps"],  "pns_ps":  s["pns_ps"],
            "tok_reduction_pct": round(tok_red, 1), "step_reduction_pct": round(step_red, 1),
            "avg_pn": s["avg_pn"], "pn_per_step": s["pn_per_step"],
            "pns_extracted_answer": pns_extracted,
            "correct": int(answers_equivalent(pns_extracted, s["answer"])),
            "orig_cot": s["orig_cot"], "pns_chain": s["pns_chain"],
        }, ensure_ascii=False) + "\n")

print(f"Saved {len(samples)} records -> {OUT_PATH}")
files.download(OUT_PATH)